In [ ]:
# Cell 1: Install arc-agi from competition wheels (offline, works in Phase A and B)
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed from competition wheels')

In [ ]:
# Cell 2: Write VericodingAgent to /tmp/my_agent.py
agent_code = """import os, sys, random, itertools
import numpy as np
from agents.agent import Agent
from arcengine import GameAction


class VericodingAgent(Agent):
    """Multi-strategy: GraphExplorer -> DenseExplorer -> beam search -> fallback."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        self._phase = 'graph'      # graph -> dense -> beam -> fallback
        self._phase_step = 0
        self._visited = set()
        self._last_grid = None
        self._last_action = None
        self._stall_count = 0
        self._prev_action = None
        self._scan_x, self._scan_y = 0, 0
        self._scan_dir = 0          # 0=right, 1=down, 2=left, 3=up
        # D4 Zobrist table
        rng = np.random.RandomState(42)
        self._zob = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)

    def _hash(self, grid):
        h, w = grid.shape[:2]
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip: g = np.fliplr(g)
                ph = np.bitwise_xor.reduce(
                    self._zob[:g.shape[0], :g.shape[1]] * (g[..., None] > 0), axis=2
                )
                hv = int(np.bitwise_xor.reduce(ph))
                if hv < best: best = hv
        return int(best)

    def choose_action(self, frames, latest_frame) -> GameAction:
        try: return self._strategy(frames, latest_frame)
        except Exception as e:
            print(f'[ERC] {e}')
            return self._random_action(latest_frame)

    def _strategy(self, frames, latest_frame) -> GameAction:
        self._step += 1
        self._phase_step += 1
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        avail = list(latest_frame.available_actions) if latest_frame.available_actions else []
        if not avail:
            return GameAction(1)

        h, w = grid.shape[:2]
        state_hash = self._hash(grid)
        is_new = state_hash not in self._visited
        self._visited.add(state_hash)

        # Stall detection
        if self._last_grid is not None and np.array_equal(grid, self._last_grid):
            self._stall_count += 1
        else:
            self._stall_count = 0
        self._last_grid = grid.copy()

        # ----- PHASE SELECTION -----
        if self._phase_step > 60 or self._stall_count > 5:
            if self._phase == 'graph':
                self._phase = 'dense'
                self._phase_step = 0
                self._scan_x, self._scan_y = 0, 0
                print(f'[phase] graph->dense step={self._step}')
            elif self._phase == 'dense':
                self._phase = 'fallback'
                self._phase_step = 0
                print(f'[phase] dense->fallback step={self._step}')

        # ----- COMPLEX (CLICK) vs SIMPLE -----
        complex_avail = [a for a in avail if a is not None and hasattr(a,'is_complex') and a.is_complex()]
        simple_avail  = [a for a in avail if a not in complex_avail]
        has_complex = len(complex_avail) > 0

        # Count non-zero for visualization
        nz = np.count_nonzero(grid)

        # ----- GRAPH PHASE: systematic click on non-zero cells -----
        if self._phase == 'graph' and has_complex:
            flat = grid.flatten()
            nonzero_idx = np.where(flat > 0)[0]
            if len(nonzero_idx) > 0:
                # Cycle through non-zero cells
                idx = nonzero_idx[self._phase_step % len(nonzero_idx)]
                y, x = divmod(int(idx), w)
                a = GameAction(6)
                a.set_data({'x': x, 'y': y})
                return a

        # ----- DENSE PHASE: grid scan -----
        if self._phase == 'dense' and has_complex:
            # Spiral scan pattern
            x, y = self._scan_x, self._scan_y
            self._scan_x += 1
            if self._scan_x >= w:
                self._scan_x = 0
                self._scan_y += 1
            if self._scan_y >= h:
                self._scan_y = 0
            # Only click if cell has content
            if 0 <= y < h and 0 <= x < w and grid[y, x] > 0:
                a = GameAction(6)
                a.set_data({'x': x, 'y': y})
                return a
            # Try next cell on next step
            return self._random_action(latest_frame)

        # ----- FALLBACK: cycle through actions -----
        # Pick action we haven't tried recently
        if self._prev_action is not None and len(simple_avail) > 1:
            others = [a for a in simple_avail if a != self._prev_action]
            if others:
                chosen = random.choice(others)
                self._prev_action = chosen
                a = GameAction(chosen)
                if a.is_complex():
                    a.set_data({'x': w//2, 'y': h//2})
                return a

        # LAST RESORT: random
        return self._random_action(latest_frame)

    def _random_action(self, latest_frame) -> GameAction:
        avail = list(getattr(latest_frame, 'available_actions', None) or [])
        if not avail: return GameAction(1)
        at = random.choice(avail)
        a = GameAction(at)
        if a.is_complex():
            a.set_data({'x': random.randint(0,63), 'y': random.randint(0,63)})
        return a
"""

with open("/tmp/my_agent.py", "w") as f:
    f.write(agent_code)
print('[Cell2] VericodingAgent written to /tmp/my_agent.py')


In [ ]:
# Cell 3: Phase B — Competition Rerun (gateway + framework)
import os, subprocess, shutil, sys

if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell3] Phase B: competition rerun detected')

    # Wait for gateway sidecar
    subprocess.run([
        'curl', '--fail', '--retry', '999', '--retry-all-errors',
        '--retry-delay', '5', '--retry-max-time', '600',
        'http://gateway:8001/api/games'
    ], check=True)
    print('[Cell3] gateway ready')

    # Copy framework from competition data
    comp = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3'
    fw_src = f'{comp}/ARC-AGI-3-Agents'
    fw_dst = '/kaggle/working/ARC-AGI-3-Agents'
    if os.path.exists(fw_src):
        shutil.copytree(fw_src, fw_dst, dirs_exist_ok=True)
    elif os.path.exists(f'{comp}/arc_agi_3_agents'):
        shutil.copytree(f'{comp}/arc_agi_3_agents', fw_dst, dirs_exist_ok=True)
    print(f'[Cell3] framework copied to {fw_dst}')

    # Install agent into framework
    agents_dir = f'{fw_dst}/agents'
    shutil.copy('/tmp/my_agent.py', f'{agents_dir}/my_agent.py')

    # Register VericodingAgent in framework __init__
    init_content = """from agents.my_agent import VericodingAgent

AVAILABLE_AGENTS = {
    "vericoding": VericodingAgent,
}
"""
    with open(f'{agents_dir}/__init__.py', 'w') as f:
        f.write(init_content)

    # Run framework
    sys.path.insert(0, fw_dst)
    result = subprocess.run(
        [sys.executable, 'main.py', '--agent', 'vericoding'],
        cwd=fw_dst, capture_output=False
    )
    print(f'[Cell3] framework exit code: {result.returncode}')
else:
    print('[Cell3] Phase A: skipping (no competition rerun)')

In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet (required to enable Submit button)
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by framework gateway')